# PathXDRP — Phase 3 (Self-Contained Kaggle Notebook)

Everything is inline. No uploads required. Just run top-to-bottom.

**Two models trained:**
- **Version A — No Mamba:** GAT drug encoder + PathwaySet cell encoder
- **Version B — With Mamba:** GraphMamba drug encoder + GeneMamba cell encoder

**Before running:** In Kaggle settings → Accelerator → `GPU T4 x2` (or P100/V100), and enable **Internet**.

**Downloads (auto):** GDSC2 IC50 (~50 MB), DepMap expression (~507 MB), DepMap Model.csv (~4 MB), Cell_Lines_Details.xlsx (~1 MB)

---
## 1. Install Packages

In [ ]:
import subprocess
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)

import torch
TORCH_VER = torch.__version__.split('+')[0]
CUDA_TAG  = (torch.version.cuda or '').replace('.', '')
print(f'PyTorch {TORCH_VER} | CUDA {torch.version.cuda} | GPUs: {torch.cuda.device_count()}')

PYG_URL = f'https://data.pyg.org/whl/torch-{TORCH_VER}+cu{CUDA_TAG}.html'
!pip install -q torch_geometric
!pip install -q pyg_lib torch_scatter torch_sparse torch_cluster -f {PYG_URL}
!pip install -q rdkit openpyxl tqdm scipy scikit-learn requests pyarrow transformers

import torch_geometric
print(f'torch_geometric {torch_geometric.__version__} OK')

In [ ]:
# -- Mamba install (CUDA only, ~10 min compile) ------------------------------
# Set INSTALL_MAMBA = False to skip and only run Version A.
INSTALL_MAMBA = True

MAMBA_READY = False
if INSTALL_MAMBA and torch.cuda.is_available():
    print('Installing causal-conv1d ...')
    !pip install -q causal-conv1d>=1.4.0
    print('Installing mamba-ssm (compiles CUDA kernels) ...')
    !pip install -q mamba-ssm --no-build-isolation
    try:
        from mamba_ssm import Mamba
        m = Mamba(d_model=64, d_state=16, d_conv=4).cuda()
        _ = m(torch.randn(2, 8, 64).cuda())
        print('mamba_ssm smoke test PASSED âœ"')
        MAMBA_READY = True
    except Exception as e:
        print(f'mamba_ssm smoke test FAILED: {e}')
else:
    print('Mamba install skipped.')
print(f'MAMBA_READY = {MAMBA_READY}')

---
## 2. Download All Data

In [ ]:
import os, json, time, requests
from pathlib import Path
from tqdm.auto import tqdm
import pandas as pd
import numpy as np

WORK = Path('/kaggle/working')
RAW  = WORK / 'data' / 'raw'
PROC = WORK / 'data' / 'processed'
for d in [RAW, PROC]: d.mkdir(parents=True, exist_ok=True)

def _download(url, dest, min_mb=0, desc=None):
    dest = Path(dest)
    if dest.exists() and dest.stat().st_size >= min_mb * 1024**2:
        print(f'  {dest.name}: already present ({dest.stat().st_size/1024**2:.0f} MB)  -  skip')
        return
    print(f'  Downloading {desc or dest.name} ...')
    hdrs = {'User-Agent': 'pathxdrp-kaggle/1.0'}
    r = requests.get(url, stream=True, timeout=600, headers=hdrs)
    r.raise_for_status()
    total = int(r.headers.get('content-length', 0))
    tmp = dest.with_suffix('.tmp')
    with open(tmp, 'wb') as f, tqdm(total=total or None, unit='B', unit_scale=True, desc=desc or dest.name) as pb:
        for chunk in r.iter_content(1 << 20):
            f.write(chunk); pb.update(len(chunk))
    tmp.rename(dest)
    print(f'  Saved {dest.name}: {dest.stat().st_size/1024**2:.0f} MB')

# -- GDSC2 IC50 data (from Sanger Institute) -------------------------------------
GDSC2_URL = 'https://cog.sanger.ac.uk/cancerrxgene/GDSC_release8.5/GDSC2_fitted_dose_response_27Oct23.xlsx'
GDSC2_XLS = RAW / 'GDSC2_fitted_dose_response.xlsx'
_download(GDSC2_URL, GDSC2_XLS, min_mb=10, desc='GDSC2 IC50')

# -- Screened compounds / drug metadata (Sanger) -------------------------
CPD_URL = 'https://cog.sanger.ac.uk/cancerrxgene/GDSC_release8.5/screened_compounds_rel_8.5.csv'
CPD_CSV = RAW / 'screened_compounds.csv'
_download(CPD_URL, CPD_CSV, min_mb=0.01, desc='screened compounds')

# -- Cell Lines Details (Sanger, for tissue labels) ----------------------
CELL_URL = 'https://cog.sanger.ac.uk/cancerrxgene/GDSC_release8.5/Cell_Lines_Details.xlsx'
CELL_XLS = RAW / 'Cell_Lines_Details.xlsx'
_download(CELL_URL, CELL_XLS, min_mb=0.05, desc='Cell Lines Details')

# -- DepMap Model.csv (COSMIC->ACH mapping) ------------------------------------------
# Use DepMap portal API with explicit release so the URL is stable.
MODEL_CSV = PROC / 'Model.csv'
if not MODEL_CSV.exists():
    _model_url = ('https://depmap.org/portal/download/api/download'
                  '?file_name=downloads-by-canonical-id%2Fpublic-26q1-5bbf.37%2FModel.csv'
                  '&dl_name=Model.csv&bucket=depmap-external-downloads')
    try:
        _download(_model_url, MODEL_CSV, desc='DepMap Model.csv')
    except Exception as _e:
        print(f'  WARNING: Model.csv download failed ({_e}). Will use SANGER_MODEL_ID fallback.')
else:
    print(f'  {MODEL_CSV.name}: already present')

# -- DepMap expression matrix (~305 MB, DepMap portal 26Q1) -------------------------
# 26Q1 renamed the file and made rows ProfileID-indexed (ModelID is a column).
EXPR_FNAME = 'OmicsExpressionTPMLogp1HumanProteinCodingGenes.csv'
EXPR_FILE  = RAW / EXPR_FNAME
_expr_url  = ('https://depmap.org/portal/download/api/download'
              '?file_name=downloads-by-canonical-id%2Fpublic-26q1-5bbf.27%2F'
              'OmicsExpressionTPMLogp1HumanProteinCodingGenes.csv'
              '&dl_name=OmicsExpressionTPMLogp1HumanProteinCodingGenes.csv'
              '&bucket=depmap-external-downloads')
_download(_expr_url, EXPR_FILE, min_mb=250, desc='DepMap expression (~305 MB)')

print('\nAll downloads complete.')


---
## 3. Preprocessing

In [ ]:
# -- 3.1  Build GDSC2 dataset CSV and compounds annotation -------------------

GDSC2_CSV = PROC / 'GDSC2-dataset.csv'
DRUGS_CSV = PROC / 'Compounds-annotation.csv'

if not GDSC2_CSV.exists():
    print('Reading GDSC2 xlsx (may take ~30 s) ...')
    gdsc2 = pd.read_excel(GDSC2_XLS, engine='openpyxl')
    gdsc2.to_csv(GDSC2_CSV, index=False)
    print(f'  Saved GDSC2 CSV: {len(gdsc2):,} rows')
else:
    gdsc2 = pd.read_csv(GDSC2_CSV)
    print(f'GDSC2 CSV: {len(gdsc2):,} rows (cached)')

if not DRUGS_CSV.exists():
    cpd = pd.read_csv(CPD_CSV)
    # Normalise columns  -  the Sanger CSV uses different casing in different releases
    cpd.columns = [c.strip().upper().replace(' ', '_') for c in cpd.columns]
    rename = {
        'DRUG_ID': 'DRUG_ID', 'DRUG_NAME': 'DRUG_NAME',
        'SYNONYMS': 'SYNONYMS', 'TARGET': 'TARGET',
        'TARGET_PATHWAY': 'TARGET_PATHWAY',
        'PUTATIVE_TARGET': 'TARGET', 'PATHWAY_NAME': 'TARGET_PATHWAY',
    }
    cpd = cpd.rename(columns={k: v for k, v in rename.items() if k in cpd.columns})
    # Make sure required columns exist
    for col in ['DRUG_ID', 'DRUG_NAME']:
        if col not in cpd.columns and col in gdsc2.columns:
            cpd = gdsc2[['DRUG_ID', 'DRUG_NAME', 'PUTATIVE_TARGET', 'PATHWAY_NAME']].drop_duplicates('DRUG_ID')
            cpd = cpd.rename(columns={'PUTATIVE_TARGET': 'TARGET', 'PATHWAY_NAME': 'TARGET_PATHWAY'})
            break
    if 'SYNONYMS' not in cpd.columns:
        cpd['SYNONYMS'] = None
    cpd[['DRUG_ID', 'DRUG_NAME', 'SYNONYMS', 'TARGET', 'TARGET_PATHWAY']].to_csv(DRUGS_CSV, index=False)
    print(f'Compounds annotation: {len(cpd)} drugs')
else:
    print(f'Compounds annotation: {len(pd.read_csv(DRUGS_CSV))} drugs (cached)')

print('Columns in GDSC2:', list(gdsc2.columns))

In [ ]:
# -- 3.2  Build COSMIC->DepMap ModelID mapping --------------------------------

COSMIC_MAP = PROC / 'cosmic_to_depmap.csv'

def _build_from_model_csv(path):
    model_df = pd.read_csv(path)
    model_df.columns = [c.strip() for c in model_df.columns]
    print(f'  Model.csv columns: {list(model_df.columns)}')
    cosmic_col = next((c for c in model_df.columns if 'COSMIC' in c.upper()), None)
    model_col  = next((c for c in model_df.columns
                       if c in ('ModelID', 'DepMap_ID', 'model_id', 'depmap_id')), None)
    if model_col is None:
        model_col = next((c for c in model_df.columns
                          if 'model' in c.lower() and 'id' in c.lower()), None)
    if cosmic_col and model_col:
        mp = model_df[[model_col, cosmic_col]].copy()
        mp.columns = ['ModelID', 'COSMICID']
        mp['COSMICID'] = pd.to_numeric(mp['COSMICID'], errors='coerce')
        mp = mp.dropna(subset=['COSMICID'])
        mp['COSMICID'] = mp['COSMICID'].astype(int)
        mp.to_csv(COSMIC_MAP, index=False)
        print(f'  cosmic_to_depmap.csv built from Model.csv: {len(mp)} entries')
        return True
    print(f'  WARNING: COSMIC/ModelID columns not found. Available: {list(model_df.columns)}')
    return False

def _build_from_sanger_id():
    """Fallback: SANGER_MODEL_ID in GDSC release 8.5 contains ACH-* DepMap IDs."""
    print('  Attempting SANGER_MODEL_ID fallback ...')
    gdsc_tmp = pd.read_csv(GDSC2_CSV, usecols=['COSMIC_ID', 'SANGER_MODEL_ID'])
    gdsc_tmp = gdsc_tmp.drop_duplicates().dropna(subset=['SANGER_MODEL_ID'])
    # Read ModelID column (26Q1 has ProfileID as row-0, ModelID as a column;
    # 24Q4 had ModelID as the first/index column — try both).
    try:
        expr_idx = pd.read_csv(EXPR_FILE, usecols=['ModelID']).iloc[:, 0].astype(str)
    except ValueError:
        expr_idx = pd.read_csv(EXPR_FILE, usecols=[0]).iloc[:, 0].astype(str)
    expr_model_set = set(expr_idx)
    print(f'  Expression matrix ModelID sample: {list(expr_idx.head(5))}')
    print(f'  SANGER_MODEL_ID sample: {list(gdsc_tmp["SANGER_MODEL_ID"].head(5))}')
    matches = gdsc_tmp[gdsc_tmp['SANGER_MODEL_ID'].isin(expr_model_set)]
    if len(matches) > 0:
        mp = matches.rename(columns={'SANGER_MODEL_ID': 'ModelID', 'COSMIC_ID': 'COSMICID'})
        mp['COSMICID'] = pd.to_numeric(mp['COSMICID'], errors='coerce')
        mp = mp.dropna(subset=['COSMICID'])
        mp['COSMICID'] = mp['COSMICID'].astype(int)
        mp.to_csv(COSMIC_MAP, index=False)
        print(f'  cosmic_to_depmap.csv built via SANGER_MODEL_ID: {len(mp)} entries')
        return True
    print('  SANGER_MODEL_ID did not match expression matrix index.')
    return False

if not COSMIC_MAP.exists():
    built = False
    if MODEL_CSV.exists():
        built = _build_from_model_csv(MODEL_CSV)
    if not built:
        built = _build_from_sanger_id()
    if not built:
        raise RuntimeError(
            'Could not build COSMIC->DepMap mapping.\n'
            'Manual fix: download Model.csv from https://depmap.org/portal/download/all/\n'
            f'and save to {MODEL_CSV}')
else:
    print(f'cosmic_to_depmap.csv: {len(pd.read_csv(COSMIC_MAP))} entries (cached)')


In [ ]:
# -- 3.3  Fetch SMILES from PubChem -----------------------------------------

SMILES_PKL = PROC / 'drugs_with_smiles.parquet'

if SMILES_PKL.exists():
    _s = pd.read_parquet(SMILES_PKL)
    print(f'SMILES already fetched: {_s["SMILES"].notna().sum()}/{len(_s)}  -  skip')
else:
    compounds = pd.read_csv(DRUGS_CSV)
    PUBCHEM = 'https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{}/property/CanonicalSMILES,IsomericSMILES/JSON'

    def _fetch_smi(name, sess):
        try:
            r = sess.get(PUBCHEM.format(requests.utils.quote(name)), timeout=15)
            if r.status_code == 200:
                props = r.json()['PropertyTable']['Properties'][0]
                for k in ('IsomericSMILES','CanonicalSMILES','SMILES'):
                    if props.get(k): return props[k]
        except: pass
        return None

    rows, sess = [], requests.Session()
    sess.headers['User-Agent'] = 'PathXDRP/1.0'
    for _, row in tqdm(compounds.iterrows(), total=len(compounds), desc='Fetching SMILES'):
        smi = _fetch_smi(str(row['DRUG_NAME']), sess)
        if smi is None and pd.notna(row.get('SYNONYMS')):
            smi = _fetch_smi(str(row['SYNONYMS']).split(',')[0].strip(), sess)
        rows.append({'DRUG_ID': row['DRUG_ID'], 'DRUG_NAME': row['DRUG_NAME'],
                     'TARGET': row.get('TARGET'), 'TARGET_PATHWAY': row.get('TARGET_PATHWAY'),
                     'SMILES': smi})
        time.sleep(0.22)
    df_s = pd.DataFrame(rows)
    df_s.to_parquet(SMILES_PKL, index=False)
    print(f'SMILES done: {df_s["SMILES"].notna().sum()}/{len(df_s)} found')

In [ ]:
# -- 3.4  Load expression matrix ---------------------------------------------

def load_expression():
    mapping = pd.read_csv(COSMIC_MAP)
    cosmic_col = 'COSMICID' if 'COSMICID' in mapping.columns else 'COSMIC_ID'
    mapping = mapping[mapping[cosmic_col].notna()].copy()
    mapping[cosmic_col] = mapping[cosmic_col].astype(int)
    model_to_cosmic = dict(zip(mapping['ModelID'], mapping[cosmic_col]))
    print(f'  Mapping: {len(model_to_cosmic)} cell lines')

    print(f'  Loading expression matrix ({EXPR_FILE.stat().st_size/1024**2:.0f} MB) ...')
    t0 = time.time()
    expr = pd.read_csv(EXPR_FILE, index_col=0)
    print(f'    -> {expr.shape[0]} rows x {expr.shape[1]:,} cols in {time.time()-t0:.1f}s')
    # 26Q1 format: index=ProfileID, columns include is_default_entry + ModelID + genes
    if 'ModelID' in expr.columns:
        if 'is_default_entry' in expr.columns:
            expr = expr[expr['is_default_entry'] == True].copy()
        expr = expr.set_index('ModelID')
        _drop = [c for c in ('is_default_entry', 'ProfileID') if c in expr.columns]
        if _drop:
            expr = expr.drop(columns=_drop)
    # Drop any remaining non-numeric columns (safety net)
    expr = expr.select_dtypes(include='number')
    expr.columns = pd.Index([c.split(' (')[0].strip() for c in expr.columns])
    print(f'    -> {expr.shape[0]} models x {expr.shape[1]:,} genes after index normalisation')

    valid = [m for m in expr.index if m in model_to_cosmic]
    expr = expr.loc[valid].copy()
    expr.index = pd.Index([model_to_cosmic[m] for m in expr.index], name='COSMIC_ID', dtype=int)

    print('  Z-scoring ...')
    mu, sd = expr.mean(0), expr.std(0)
    sd[sd == 0] = 1.0
    expr = (expr - mu) / sd
    print(f'  Expression ready: {expr.shape}')
    return expr.astype('float32')

expr_matrix = load_expression()

In [ ]:
# -- 3.5  Build master DataFrame ---------------------------------------------

def load_cell_metadata():
    try:
        cells = pd.read_excel(CELL_XLS)
        rename = {
            'COSMIC identifier': 'COSMIC_ID',
            'Sample Name': 'CELL_LINE_NAME',
            'GDSC\nTissue descriptor 1': 'tissue_1',
            'GDSC\nTissue\ndescriptor 2': 'tissue_2',
            'Cancer Type\n(matching TCGA label)': 'cancer_type',
            'Microsatellite \ninstability Status (MSI)': 'msi_status',
        }
        cells = cells.rename(columns={k: v for k, v in rename.items() if k in cells.columns})
        cells['COSMIC_ID'] = pd.to_numeric(cells['COSMIC_ID'], errors='coerce')
        return cells
    except Exception as e:
        print(f'  Cell metadata load failed ({e}); tissue columns will be absent')
        return pd.DataFrame(columns=['COSMIC_ID','tissue_1','tissue_2','cancer_type','msi_status'])

response = pd.read_csv(GDSC2_CSV).drop_duplicates(subset=['DRUG_ID','COSMIC_ID'])
drugs    = pd.read_parquet(SMILES_PKL)
cells    = load_cell_metadata()

df = response.merge(drugs[['DRUG_ID','SMILES','TARGET','TARGET_PATHWAY']], on='DRUG_ID', how='left')
if not cells.empty:
    df = df.merge(cells[['COSMIC_ID','tissue_1','tissue_2','cancer_type','msi_status']],
                  on='COSMIC_ID', how='left')
df = df[df['SMILES'].notna()].copy()
cells_with_expr = set(expr_matrix.index)
df = df[df['COSMIC_ID'].isin(cells_with_expr)].reset_index(drop=True)

print(f'Master DF: {len(df):,} rows | {df["DRUG_ID"].nunique()} drugs | {df["COSMIC_ID"].nunique()} cells')

In [ ]:
# -- 3.6  Build KEGG pathway -> gene map ------------------------------------

PGM_PATH = PROC / 'pathway_gene_map.json'

if PGM_PATH.exists():
    with open(PGM_PATH) as f: pathway_gene_symbols = json.load(f)
    print(f'pathway_gene_map.json: {len(pathway_gene_symbols)} pathways (cached)')
else:
    KEGG = 'https://rest.kegg.jp'
    DELAY = 0.4

    def _kegg_get(url, desc):
        time.sleep(DELAY)
        for attempt in range(3):
            try:
                r = requests.get(url, timeout=120, headers={'User-Agent':'pathxdrp/1.0'})
                r.raise_for_status(); return r.text
            except Exception as e:
                print(f'  [{desc}] attempt {attempt+1}/3: {e}')
                time.sleep(2**attempt)
        raise RuntimeError(f'KEGG fetch failed: {url}')

    print('Fetching KEGG pathway list ...')
    pw_text = _kegg_get(f'{KEGG}/list/pathway/hsa', 'pathway list')
    pathway_names = {}
    for line in pw_text.strip().split('\n'):
        if '\t' not in line: continue
        pid, name = line.split('\t', 1)
        pathway_names[pid.replace('path:', '')] = name.split(' - Homo sapiens')[0].strip()
    print(f'  {len(pathway_names)} human pathways')

    print('Fetching KEGG gene-pathway links (~30 s) ...')
    from collections import defaultdict
    gp_text = _kegg_get(f'{KEGG}/link/pathway/hsa', 'gene-pathway')
    pathway_entrez = defaultdict(list)
    for line in gp_text.strip().split('\n'):
        if '\t' not in line: continue
        gpart, ppart = line.strip().split('\t', 1)
        try: eid = int(gpart.replace('hsa:', ''))
        except ValueError: continue
        pathway_entrez[ppart.replace('path:', '')].append(eid)

    # Parse entrez->symbol from expression matrix column names ("TSPAN6 (7105)" format)
    print('Building entrez->symbol map from expression matrix header ...')
    hdr = pd.read_csv(EXPR_FILE, nrows=0, index_col=0)
    expr_gene_set = set(c.split(' (')[0].strip() for c in hdr.columns)
    entrez_to_sym = {}
    for col in hdr.columns:
        if ' (' in col and col.endswith(')'):
            sym = col.split(' (')[0].strip()
            try:
                eid = int(col.split('(')[1].rstrip(')').strip())
                if sym in expr_gene_set: entrez_to_sym[eid] = sym
            except ValueError: pass
    print(f'  {len(entrez_to_sym):,} entrez->symbol mappings')

    # Build pathway->[gene_symbols]
    pathway_gene_symbols = {}
    for pid, eids in pathway_entrez.items():
        if pid not in pathway_names: continue
        seen, syms = set(), []
        for eid in eids:
            s = entrez_to_sym.get(eid)
            if s and s not in seen: seen.add(s); syms.append(s)
        if syms: pathway_gene_symbols[pathway_names[pid]] = syms

    with open(PGM_PATH, 'w') as f: json.dump(pathway_gene_symbols, f)
    print(f'pathway_gene_map.json: {len(pathway_gene_symbols)} pathways saved')

In [ ]:
# -- 3.7  Build train/val/test splits --------------------------------------
import hashlib
from sklearn.model_selection import GroupKFold, KFold

SPLITS_DIR = PROC / 'splits'
N_FOLDS = 5

def _df_hash(df):
    return hashlib.md5((str(sorted(df.columns.tolist())) + str(len(df))).encode()).hexdigest()[:8]

def _save_fold(d, tr, va, te):
    Path(d).mkdir(parents=True, exist_ok=True)
    for name, arr in [('train', tr), ('val', va), ('test', te)]:
        np.save(str(Path(d) / f'{name}.npy'), np.array(arr))

def _random_split(df, seed=0):
    folds = []
    for tr, te in KFold(N_FOLDS, shuffle=True, random_state=seed).split(np.arange(len(df))):
        rng = np.random.default_rng(seed); rng.shuffle(te)
        folds.append({'train': tr, 'val': te[:len(te)//2], 'test': te[len(te)//2:]})
    return folds

def _group_split(df, col, seed=0):
    folds = []
    for tr, te in GroupKFold(N_FOLDS).split(np.arange(len(df)), groups=df[col].values):
        rng = np.random.default_rng(seed); rng.shuffle(te)
        folds.append({'train': tr, 'val': te[:len(te)//2], 'test': te[len(te)//2:]})
    return folds

def _tissue_split(df, seed=0):
    if 'tissue_2' not in df.columns: return _random_split(df, seed)
    tissues = df['tissue_2'].fillna('unknown').values
    top5 = df['tissue_2'].value_counts().index[:N_FOLDS].tolist()
    folds = []
    for t in top5:
        te = np.where(tissues == t)[0]; tr = np.where(tissues != t)[0]
        rng = np.random.default_rng(seed); rng.shuffle(te)
        folds.append({'train': tr, 'val': te[:len(te)//2], 'test': te[len(te)//2:]})
    return folds

BUILDERS = {
    'random':         lambda df, s: _random_split(df, s),
    'cell_blind':     lambda df, s: _group_split(df, 'COSMIC_ID', s),
    'drug_blind':     lambda df, s: _group_split(df, 'DRUG_ID', s),
    'tissue_blind':   lambda df, s: _tissue_split(df, s),
}

def load_split(name, seed, fold):
    d = SPLITS_DIR / name / f'seed{seed}' / f'fold{fold}'
    return np.load(d/'train.npy'), np.load(d/'val.npy'), np.load(d/'test.npy')

dfhash = _df_hash(df)
for name, builder in BUILDERS.items():
    for seed in range(5):
        lock = SPLITS_DIR / name / f'seed{seed}' / 'meta.json'
        if lock.exists():
            saved = json.loads(lock.read_text())
            if saved.get('df_hash') == dfhash: continue
        folds = builder(df, seed)
        for i, fold in enumerate(folds):
            _save_fold(SPLITS_DIR / name / f'seed{seed}' / f'fold{i}',
                       fold['train'], fold['val'], fold['test'])
        (SPLITS_DIR / name / f'seed{seed}').mkdir(parents=True, exist_ok=True)
        lock.write_text(json.dumps({'df_hash': dfhash, 'name': name, 'seed': seed}))
print('All splits built.')

---
## 4. Model Definitions (Inlined)

In [ ]:
# -- 4.1  Molecular graph utilities -----------------------------------------
import warnings
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Optional
from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem, rdFingerprintGenerator
from torch_geometric.data import Data, Batch

RDLogger.DisableLog('rdApp.*')
warnings.filterwarnings('ignore', message='.*torch-scatter.*')

ATOM_TYPES = ['C','N','O','S','F','Si','P','Cl','Br','Mg','Na','Ca','Fe','As','Al','I','B','V','K',
              'Tl','Yb','Sb','Sn','Ag','Pd','Co','Se','Ti','Zn','H','Li','Ge','Cu','Au','Ni','Cd',
              'In','Mn','Zr','Cr','Pt','Hg','Pb']
HYBRIDISATION = [Chem.rdchem.HybridizationType.SP, Chem.rdchem.HybridizationType.SP2,
                 Chem.rdchem.HybridizationType.SP3, Chem.rdchem.HybridizationType.SP3D,
                 Chem.rdchem.HybridizationType.SP3D2]
BOND_TYPES = [Chem.rdchem.BondType.SINGLE, Chem.rdchem.BondType.DOUBLE,
              Chem.rdchem.BondType.TRIPLE, Chem.rdchem.BondType.AROMATIC]
MORGAN_RADIUS, MORGAN_BITS = 2, 256
_MORGAN_GEN = rdFingerprintGenerator.GetMorganGenerator(radius=MORGAN_RADIUS, fpSize=MORGAN_BITS)

def _one_hot(val, vocab):
    v = [0]*(len(vocab)+1); v[vocab.index(val) if val in vocab else len(vocab)] = 1; return v

def atom_features(atom, fg_vocab=None):
    sym = atom.GetSymbol()
    deg = [0]*11; deg[min(atom.GetDegree(),10)] = 1
    chg = [0]*7;  chg[min(max(atom.GetFormalCharge()+3,0),6)] = 1
    hyb = [0]*(len(HYBRIDISATION)+1)
    hyb[{h:i for i,h in enumerate(HYBRIDISATION)}.get(atom.GetHybridization(), len(HYBRIDISATION))] = 1
    chi = [0,0,0]
    ct = atom.GetChiralTag()
    if ct == Chem.rdchem.ChiralType.CHI_TETRAHEDRAL_CW: chi[0]=1
    elif ct == Chem.rdchem.ChiralType.CHI_TETRAHEDRAL_CCW: chi[1]=1
    else: chi[2]=1
    nhs = [0]*5; nhs[min(atom.GetTotalNumHs(),4)] = 1
    feats = _one_hot(sym,ATOM_TYPES) + deg + chg + hyb + [int(atom.GetIsAromatic())] + [int(atom.IsInRing())] + chi + nhs
    if fg_vocab is not None:
        fg_vec = [0]*len(fg_vocab)
        mol = atom.GetOwningMol(); ao = rdFingerprintGenerator.AdditionalOutput(); ao.AllocateBitInfoMap()
        _MORGAN_GEN.GetFingerprint(mol, additionalOutput=ao)
        bm = ao.GetBitInfoMap()
        for bit, origins in (bm.items() if bm else []):
            for center, radius in origins:
                if center == atom.GetIdx() and radius == MORGAN_RADIUS and bit in fg_vocab:
                    fg_vec[fg_vocab[bit]] = 1
        feats += fg_vec
    return feats

def bond_features(bond):
    bt = bond.GetBondType()
    bt_oh = [0]*(len(BOND_TYPES)+1); bt_oh[{b:i for i,b in enumerate(BOND_TYPES)}.get(bt,len(BOND_TYPES))] = 1
    stereo = bond.GetStereo()
    st = [0,0,0]
    if stereo == Chem.rdchem.BondStereo.STEREOE: st[0]=1
    elif stereo == Chem.rdchem.BondStereo.STEREOZ: st[1]=1
    else: st[2]=1
    return bt_oh + [int(bond.GetIsConjugated())] + [int(bond.IsInRing())] + st

def smiles_to_graph(smiles, fg_vocab=None, label=None):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return None
    x = torch.tensor([atom_features(a, fg_vocab) for a in mol.GetAtoms()], dtype=torch.float)
    es, ed, ea = [], [], []
    for b in mol.GetBonds():
        i,j = b.GetBeginAtomIdx(), b.GetEndAtomIdx(); bf = bond_features(b)
        for u,v in [(i,j),(j,i)]: es.append(u); ed.append(v); ea.append(bf)
    if not es:
        ei = torch.zeros((2,0),dtype=torch.long)
        eattr = torch.zeros((0,9),dtype=torch.float)
    else:
        ei = torch.tensor([es,ed],dtype=torch.long)
        eattr = torch.tensor(ea,dtype=torch.float)
    g = Data(x=x, edge_index=ei, edge_attr=eattr)
    if label is not None: g.y = torch.tensor([label],dtype=torch.float)
    return g

def build_fg_vocab(smiles_list, top_k=MORGAN_BITS):
    from collections import Counter
    cnt = Counter()
    gen = rdFingerprintGenerator.GetMorganGenerator(radius=MORGAN_RADIUS, fpSize=top_k)
    for smi in smiles_list:
        mol = Chem.MolFromSmiles(smi)
        if mol is None: continue
        ao = rdFingerprintGenerator.AdditionalOutput(); ao.AllocateBitInfoMap()
        gen.GetFingerprint(mol, additionalOutput=ao)
        bm = ao.GetBitInfoMap()
        if bm: cnt.update(bm.keys())
    return {bit: idx for idx,(bit,_) in enumerate(cnt.most_common(top_k))}

print('Graph utilities loaded.')

In [ ]:
# -- 4.2  Drug GAT encoder --------------------------------------------------
from torch_geometric.nn import GATv2Conv, global_mean_pool, global_max_pool

class DrugGATEncoder(nn.Module):
    def __init__(self, node_in_dim, edge_in_dim, hidden_dim=128, n_layers=3,
                 n_heads=8, dropout=0.1, use_molformer=False):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.node_proj = nn.Linear(node_in_dim, hidden_dim)
        self.convs = nn.ModuleList([
            GATv2Conv(hidden_dim, hidden_dim//n_heads, heads=n_heads,
                      edge_dim=edge_in_dim, concat=True, dropout=dropout)
            for _ in range(n_layers)])
        self.norms = nn.ModuleList([nn.LayerNorm(hidden_dim) for _ in range(n_layers)])
        self.dropout = nn.Dropout(dropout)
        self.act = nn.GELU()

    def forward(self, data, smiles_list=None, batch=None):
        x = self.act(self.node_proj(data.x))
        for conv, norm in zip(self.convs, self.norms):
            x = norm(x + self.dropout(conv(x, data.edge_index, data.edge_attr)))
            x = self.act(x)
        if batch is None:
            batch = data.batch if hasattr(data,'batch') and data.batch is not None \
                    else torch.zeros(x.size(0), dtype=torch.long, device=x.device)
        # h_mol: (B, 2*hidden_dim)  -  mean+max concat for richer global readout
        h_mol = torch.cat([global_mean_pool(x, batch), global_max_pool(x, batch)], dim=-1)
        return x, h_mol  # (N_atoms, D), (B, 2D)

print('DrugGATEncoder defined.')

In [ ]:
# -- 4.3  Graph-Mamba drug encoder (requires mamba_ssm) ---------------------
from torch_geometric.utils import degree, to_dense_batch

try:
    from mamba_ssm import Mamba as _MambaCls
    MAMBA_PKG = True
except Exception:
    _MambaCls = None
    MAMBA_PKG = False

class _BiMambaBlock(nn.Module):
    def __init__(self, hidden_dim, d_state=16, d_conv=4):
        super().__init__()
        if not MAMBA_PKG: raise ImportError('mamba_ssm not installed')
        self.fwd  = _MambaCls(d_model=hidden_dim, d_state=d_state, d_conv=d_conv)
        self.bwd  = _MambaCls(d_model=hidden_dim, d_state=d_state, d_conv=d_conv)
        self.gate = nn.Linear(hidden_dim*2, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)

    def forward(self, x, pad_mask):
        x_in = x * pad_mask.unsqueeze(-1).float()
        h_f = self.fwd(x_in)
        h_b = torch.flip(self.bwd(torch.flip(x_in, dims=[1])), dims=[1])
        return self.norm(x + self.gate(torch.cat([h_f, h_b], dim=-1)))

class GraphMambaDrugEncoder(nn.Module):
    def __init__(self, node_in_dim, edge_in_dim, hidden_dim=128,
                 n_gat_layers=2, n_mamba_layers=2, n_heads=8, dropout=0.1, ordering='degree'):
        super().__init__()
        if not MAMBA_PKG: raise ImportError('mamba_ssm not installed')
        self.hidden_dim = hidden_dim
        self.ordering = ordering
        self.node_proj = nn.Linear(node_in_dim, hidden_dim)
        self.gat_convs = nn.ModuleList([
            GATv2Conv(hidden_dim, hidden_dim//n_heads, heads=n_heads,
                      edge_dim=edge_in_dim, concat=True, dropout=dropout)
            for _ in range(n_gat_layers)])
        self.gat_norms = nn.ModuleList([nn.LayerNorm(hidden_dim) for _ in range(n_gat_layers)])
        self.mamba_blocks = nn.ModuleList([_BiMambaBlock(hidden_dim) for _ in range(n_mamba_layers)])
        self.dropout = nn.Dropout(dropout)
        self.act = nn.GELU()

    def _atom_order(self, edge_index, batch_idx):
        n = batch_idx.numel()
        if self.ordering == 'canonical': return torch.arange(n, device=batch_idx.device)
        deg = degree(edge_index[0], num_nodes=n).to(batch_idx.device)
        order = torch.empty(n, dtype=torch.long, device=batch_idx.device)
        for b in batch_idx.unique():
            mask = batch_idx == b; idx = torch.nonzero(mask, as_tuple=False).squeeze(-1)
            order[mask] = idx[torch.argsort(deg[idx], descending=True)]
        return order

    def forward(self, data, smiles_list=None, batch=None):
        if batch is None:
            batch = data.batch if hasattr(data,'batch') and data.batch is not None \
                    else torch.zeros(data.x.size(0), dtype=torch.long, device=data.x.device)
        x = self.act(self.node_proj(data.x))
        for conv, norm in zip(self.gat_convs, self.gat_norms):
            x = norm(x + self.dropout(conv(x, data.edge_index, data.edge_attr)))
            x = self.act(x)
        perm = self._atom_order(data.edge_index, batch)
        inv_perm = torch.empty_like(perm); inv_perm[perm] = torch.arange(perm.numel(), device=perm.device)
        x_pad, pad_m = to_dense_batch(x[perm], batch[perm])
        for blk in self.mamba_blocks: x_pad = blk(x_pad, pad_m)
        x_perm_out = x_pad[pad_m]
        x_out = torch.empty_like(x_perm_out); x_out[perm] = x_perm_out
        # Return (B, 2D) h_mol to match DrugGATEncoder interface
        h_mol = torch.cat([global_mean_pool(x_out, batch), global_max_pool(x_out, batch)], dim=-1)
        return x_out, h_mol

print(f'GraphMambaDrugEncoder defined (MAMBA_PKG={MAMBA_PKG}).')

In [ ]:
# -- 4.4  PathwaySet cell encoder -------------------------------------------

class PathwaySetEncoder(nn.Module):
    def __init__(self, n_genes, pathway_gene_map, hidden_dim=128,
                 dropout=0.1, n_pw_transformer_layers=1):
        super().__init__()
        self.pathway_names = sorted(pathway_gene_map.keys())
        self.n_pathways = len(self.pathway_names)
        self.hidden_dim = hidden_dim
        self.gene_proj = nn.Sequential(nn.Linear(3, hidden_dim), nn.GELU(), nn.LayerNorm(hidden_dim))
        _nh = max(1, hidden_dim//32)
        self.pathway_transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=hidden_dim, nhead=_nh,
                dim_feedforward=hidden_dim*2, dropout=0.0,
                batch_first=True, norm_first=True),
            num_layers=n_pw_transformer_layers, enable_nested_tensor=False
        ) if n_pw_transformer_layers > 0 else None
        self.pathway_norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)

        # Pre-compute averaging matrix
        flat_gene, flat_pw = [], []
        for p_idx, pname in enumerate(self.pathway_names):
            for g in pathway_gene_map[pname]:
                flat_gene.append(g); flat_pw.append(p_idx)
        pw_sizes = torch.tensor([float(max(len(pathway_gene_map[p]),1)) for p in self.pathway_names])
        avg_mat = torch.zeros(self.n_pathways, max(len(flat_gene),1))
        for i,(pw,_) in enumerate(zip(flat_pw, flat_gene)):
            avg_mat[pw, i] = 1.0 / pw_sizes[pw].item()
        self.register_buffer('avg_matrix', avg_mat)
        self.register_buffer('flat_gene_idx',
            torch.tensor(flat_gene, dtype=torch.long) if flat_gene else torch.zeros(0, dtype=torch.long))
        self._n_pairs = len(flat_gene)

    def forward(self, expr):
        B = expr.size(0)
        if self._n_pairs == 0:
            return self.dropout(torch.zeros(B, self.n_pathways, self.hidden_dim, device=expr.device))
        ge = expr[:, self.flat_gene_idx].to(self.avg_matrix.dtype)
        mean_pw  = torch.matmul(ge,       self.avg_matrix.T)
        std_pw   = (torch.matmul(ge**2,   self.avg_matrix.T) - mean_pw**2).clamp(min=0).sqrt()
        frac_pw  = torch.matmul((ge > 0).to(self.avg_matrix.dtype), self.avg_matrix.T)
        h = self.gene_proj(torch.stack([mean_pw, std_pw, frac_pw], dim=-1))
        if self.pathway_transformer is not None: h = self.pathway_transformer(h)
        return self.dropout(self.pathway_norm(h))

print('PathwaySetEncoder defined.')

In [ ]:
# -- 4.5  GeneMamba cell encoder (requires mamba_ssm + HuggingFace) ---------
try:
    from transformers import AutoModel, AutoTokenizer
    HF_AVAILABLE = True
except Exception:
    AutoModel = AutoTokenizer = None
    HF_AVAILABLE = False

class _GeneBiMamba(nn.Module):
    """Standalone Bi-Mamba for the GeneMamba fallback path."""
    def __init__(self, hidden_dim, d_state=16, d_conv=4):
        super().__init__()
        if not MAMBA_PKG: raise ImportError('mamba_ssm not installed')
        self.fwd = _MambaCls(d_model=hidden_dim, d_state=d_state, d_conv=d_conv)
        self.bwd = _MambaCls(d_model=hidden_dim, d_state=d_state, d_conv=d_conv)
        self.gate = nn.Linear(hidden_dim*2, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
    def forward(self, x):
        h_b = torch.flip(self.bwd(torch.flip(x, [1])), [1])
        return self.norm(x + self.gate(torch.cat([self.fwd(x), h_b], -1)))

class GeneMambaCellEncoder(nn.Module):
    HF_DEFAULT = 'mineself2016/GeneMamba'
    def __init__(self, n_genes, gene_symbols, pathway_gene_map, hidden_dim=128,
                 top_k=2048, backbone_id=HF_DEFAULT, freeze_backbone=True,
                 n_layers_fallback=4, backbone_dim_fallback=256):
        super().__init__()
        if not MAMBA_PKG: raise ImportError('mamba_ssm not installed')
        self.top_k = min(top_k, n_genes)
        self.gene_symbols = list(gene_symbols)
        self.pathway_names = sorted(pathway_gene_map.keys())
        self.n_pathways = len(self.pathway_names)
        self.pathway_gene_map = pathway_gene_map
        self.n_genes = n_genes
        self.hidden_dim = hidden_dim

        # Load backbone
        backbone, bk_dim = self._load_backbone(backbone_id, n_layers_fallback, backbone_dim_fallback)
        self.backbone = backbone
        self.backbone_dim = bk_dim
        if freeze_backbone:
            for p in self.backbone.parameters(): p.requires_grad_(False)

        self.register_buffer('gene_token_ids', torch.full((n_genes,), -1, dtype=torch.long))
        self._token_map_init = False
        self.gene_adapter = nn.Sequential(nn.Linear(bk_dim, hidden_dim), nn.GELU(), nn.LayerNorm(hidden_dim))
        self.pathway_norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(0.1)

    def _load_backbone(self, backbone_id, n_fallback, dim_fallback):
        if HF_AVAILABLE:
            try:
                model = AutoModel.from_pretrained(backbone_id, trust_remote_code=True)
                dim = getattr(model.config,'hidden_size',None) or getattr(model.config,'d_model',dim_fallback)
                return model, int(dim)
            except Exception as e:
                print(f'[GeneMamba] HF load failed ({e}); using fallback.')
        stack = nn.ModuleList([_GeneBiMamba(dim_fallback) for _ in range(n_fallback)])
        return stack, dim_fallback

    def setup_token_map(self):
        if not HF_AVAILABLE:
            ids = torch.tensor([abs(hash(g)) % 25000 for g in self.gene_symbols], dtype=torch.long)
        else:
            try:
                tok = AutoTokenizer.from_pretrained(self.HF_DEFAULT, trust_remote_code=True)
                vocab = tok.get_vocab() if hasattr(tok,'get_vocab') else {}
                ids = torch.tensor([vocab.get(g, vocab.get(g.upper(), -1)) for g in self.gene_symbols], dtype=torch.long)
                n_ok = int((ids >= 0).sum())
                print(f'[GeneMamba] {n_ok}/{self.n_genes} genes mapped to backbone vocab.')
            except Exception as e:
                print(f'[GeneMamba] Tokenizer failed ({e}); using hash fallback.')
                ids = torch.tensor([abs(hash(g)) % 25000 for g in self.gene_symbols], dtype=torch.long)
        self.gene_token_ids.copy_(ids)
        self._token_map_init = True

    def forward(self, expr):
        if not self._token_map_init: self.setup_token_map()
        _, topk_idx = expr.topk(self.top_k, dim=1)
        gene_emb = self._encode_topk(topk_idx)
        gene_emb = self.gene_adapter(gene_emb)
        return self.dropout(self._pool_pathways(gene_emb, topk_idx, expr.device))

    def _encode_topk(self, topk_idx):
        token_ids = self.gene_token_ids[topk_idx].clamp(min=0)
        if isinstance(self.backbone, nn.ModuleList):
            if not hasattr(self, '_fb_emb'):
                vocab = max(int(self.gene_token_ids.max().item())+1, 25000)
                self._fb_emb = nn.Embedding(vocab, self.backbone_dim).to(token_ids.device)
            x = self._fb_emb(token_ids)
            for blk in self.backbone: x = blk(x)
            return x
        with torch.no_grad():
            out = self.backbone(input_ids=token_ids)
        h = getattr(out,'last_hidden_state',None)
        return h if h is not None else out[0]

    def _pool_pathways(self, gene_emb, topk_idx, device):
        B, K, D = gene_emb.shape
        out = gene_emb.new_zeros(B, self.n_pathways, D)
        if not hasattr(self, '_pw_mask'):
            mask = torch.zeros(self.n_pathways, self.n_genes, dtype=torch.bool)
            for p_i, pn in enumerate(self.pathway_names):
                for g in self.pathway_gene_map[pn]: mask[p_i, g] = True
            self.register_buffer('_pw_mask', mask, persistent=False)
        belong = self._pw_mask.to(device)[:, topk_idx].permute(1,0,2).float()  # (B,P,K)
        counts = belong.sum(-1, keepdim=True).clamp(min=1.0)
        out = torch.einsum('bpk,bkd->bpd', belong, gene_emb) / counts
        return self.pathway_norm(out)

print(f'GeneMambaCellEncoder defined (HF_AVAILABLE={HF_AVAILABLE}).')

In [ ]:
# -- 4.6  Cross-attention + Evidential head + PathXDRP ---------------------
import math
from torch_geometric.nn import global_add_pool

class PathwayMaskedCrossAttention(nn.Module):
    def __init__(self, hidden_dim=128, n_heads=8, dropout=0.1,
                 mask_type='soft', entropy_reg_weight=0.01, n_pathways=0):
        super().__init__()
        assert hidden_dim % n_heads == 0
        self.n_heads = n_heads; self.head_dim = hidden_dim//n_heads
        self.scale = math.sqrt(self.head_dim)
        self.mask_type = mask_type; self.entropy_reg_weight = entropy_reg_weight
        self.hidden_dim = hidden_dim
        self.q_proj  = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.k_proj  = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.v_proj  = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.out_proj = nn.Linear(hidden_dim, hidden_dim)
        self.norm_q  = nn.LayerNorm(hidden_dim); self.norm_kv = nn.LayerNorm(hidden_dim)
        self.attn_drop = nn.Dropout(dropout)
        if mask_type == 'soft' and n_pathways > 0:
            self.soft_mask_logit = nn.Parameter(torch.zeros(1,1,1,n_pathways))
        self._last_attn = None; self._entropy_loss = torch.tensor(0.0)

    def forward(self, h_drug, h_cell, atom_batch, hard_mask=None):
        B, N_pw = h_cell.size(0), h_cell.size(1)
        Q = self.q_proj(self.norm_q(h_drug))
        K = self.k_proj(self.norm_kv(h_cell)); V = self.v_proj(h_cell)
        Q_pad, pad_mask = to_dense_batch(Q, atom_batch)
        max_n = Q_pad.size(1)
        def to_mh(x, seq): return x.view(x.size(0),seq,self.n_heads,self.head_dim).permute(0,2,1,3)
        Q_mh = to_mh(Q_pad, max_n); K_mh = to_mh(K, N_pw); V_mh = to_mh(V, N_pw)
        scores = torch.einsum('bhnd,bhpd->bhnp', Q_mh, K_mh) / self.scale
        scores = scores.masked_fill((~pad_mask).unsqueeze(1).unsqueeze(-1), -1e4)
        if self.mask_type == 'soft' and hasattr(self, 'soft_mask_logit'):
            scores = scores + self.soft_mask_logit
        attn = self.attn_drop(F.softmax(scores, dim=-1))
        ctx = self.out_proj(
            (torch.einsum('bhnp,bhpd->bhnd', attn, V_mh)
             .permute(0,2,1,3).contiguous().view(B, max_n, self.hidden_dim))
        )[pad_mask]
        attn_flat = attn.mean(1)[pad_mask]
        p = attn_flat + 1e-9
        self._entropy_loss = -(p * p.log()).sum(-1).mean()
        self._last_attn = attn_flat.detach()
        return ctx, attn_flat


class EvidentialRegressionHead(nn.Module):
    def __init__(self, in_dim, hidden_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim), nn.GELU(), nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, 4))
    def forward(self, z):
        out = self.net(z)
        with torch.amp.autocast(device_type=z.device.type, enabled=False):
            out = out.float()
            out = out.clamp(-20, 20)  # prevent extreme raw outputs
            gamma = out[:,0]; nu = F.softplus(out[:,1]).clamp(min=0.05)+0.05
            alpha = F.softplus(out[:,2]).clamp(min=0.01)+1.1
            beta  = F.softplus(out[:,3]).clamp(min=1e-4)+1e-4
            aleat = (beta/(alpha-1)).clamp(max=1e4)
            epist = (beta/(nu*(alpha-1))).clamp(max=1e4)
        return {'mu':gamma,'nu':nu,'alpha':alpha,'beta':beta,'pred':gamma,'aleatoric':aleat,'epistemic':epist}

def evidential_loss(pred, y, lam=0.1):
    with torch.amp.autocast(device_type=y.device.type, enabled=False):
        g,nu,al,be,yf = pred['mu'].float(),pred['nu'].float(),pred['alpha'].float(),pred['beta'].float(),y.float()
        tbl = 2*be*(1+nu)
        nu_s = nu.clamp(min=1e-6); tbl_s = tbl.clamp(min=1e-6)
        nll = (0.5*torch.log(torch.tensor(torch.pi,device=y.device)/nu_s)
               - al*torch.log(tbl_s) + (al+0.5)*torch.log((nu_s*(yf-g)**2+tbl_s).clamp(min=1e-6))
               + torch.lgamma(al) - torch.lgamma(al+0.5))
        reg = torch.abs(yf-g) * (2*nu+al)
    return (nll + lam*reg).mean()


class PathXDRP(nn.Module):
    def __init__(self, node_in_dim, edge_in_dim, n_genes, pathway_gene_map,
                 hidden_dim=128, n_gat_layers=3, n_attn_heads=8, dropout=0.1,
                 mask_type='soft', entropy_reg_weight=0.01, evidential_lam=0.01,
                 drug_encoder_type='gat', cell_encoder_type='pathway_set',
                 gene_symbols=None, graph_mamba_kwargs=None, gene_mamba_kwargs=None,
                 n_pw_transformer_layers=1):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.evidential_lam = evidential_lam
        self.entropy_reg_weight = entropy_reg_weight
        self.drug_encoder_type = drug_encoder_type

        if drug_encoder_type == 'graph_mamba':
            gm = dict(graph_mamba_kwargs or {})
            self.drug_enc = GraphMambaDrugEncoder(
                node_in_dim, edge_in_dim, hidden_dim, n_heads=n_attn_heads, dropout=dropout, **gm)
        else:
            self.drug_enc = DrugGATEncoder(
                node_in_dim, edge_in_dim, hidden_dim, n_gat_layers, n_attn_heads, dropout)

        if cell_encoder_type == 'gene_mamba':
            gm2 = dict(gene_mamba_kwargs or {})
            self.cell_enc = GeneMambaCellEncoder(
                n_genes, gene_symbols, pathway_gene_map, hidden_dim, **gm2)
        else:
            self.cell_enc = PathwaySetEncoder(
                n_genes, pathway_gene_map, hidden_dim, dropout, n_pw_transformer_layers)

        self.cross_attn = PathwayMaskedCrossAttention(
            hidden_dim, n_attn_heads, dropout, mask_type, entropy_reg_weight, len(pathway_gene_map))
        self.pool_proj = nn.Linear(hidden_dim, hidden_dim)
        self.pool_norm = nn.LayerNorm(hidden_dim)
        self.cell_pool_q = nn.Parameter(torch.randn(1,1,hidden_dim)*0.02)
        # in_dim = D (drug_context) + 2D (h_mol from mean+max pool) + D (cell_global) + D (interaction) = 5D
        self.head = EvidentialRegressionHead(in_dim=hidden_dim*5, hidden_dim=hidden_dim)

    def forward(self, drug_batch, expr, smiles_list=None, hard_mask=None, y=None):
        h_atom, h_mol = self.drug_enc(drug_batch, smiles_list=smiles_list, batch=drug_batch.batch)
        h_cell = self.cell_enc(expr)
        context, attn_w = self.cross_attn(h_atom, h_cell, drug_batch.batch, hard_mask)
        a_w = attn_w.max(-1, keepdim=True)[0]
        h_drug_ctx = self.pool_norm(self.pool_proj(global_add_pool(context*a_w, drug_batch.batch)))
        pool_s = torch.matmul(
            self.cell_pool_q.expand(h_cell.size(0),-1,-1), h_cell.transpose(-1,-2)
        ) / math.sqrt(self.hidden_dim)
        h_cell_g = (F.softmax(pool_s, dim=-1) @ h_cell).squeeze(1)
        z = torch.cat([h_drug_ctx, h_mol, h_cell_g, h_drug_ctx*h_cell_g], dim=-1)
        pred = self.head(z)
        out = {'pred': pred, 'attn_weights': attn_w}
        if y is not None:
            ml = evidential_loss(pred, y, lam=self.evidential_lam)
            el = self.cross_attn._entropy_loss * self.entropy_reg_weight
            out['loss'] = ml + el; out['main_loss'] = ml; out['entropy_loss'] = el
        return out

print('PathXDRP model defined.')

In [ ]:
# -- 4.7  Metrics ------------------------------------------------------------
from scipy import stats
from sklearn.metrics import mean_squared_error, r2_score

def regression_report(y_true, y_pred, drug_ids=None, cell_ids=None, uncertainties=None):
    # Guard against NaN predictions (numerical instability during training)
    valid = np.isfinite(y_pred) & np.isfinite(y_true)
    nan_frac = (~valid).mean()
    if nan_frac > 0:
        print(f'  WARNING: {(~valid).sum()} / {len(valid)} non-finite predictions ')
        y_pred = y_pred[valid]; y_true = y_true[valid]
        if drug_ids is not None: drug_ids = drug_ids[valid]
        if uncertainties is not None: uncertainties = uncertainties[valid]
    if len(y_pred) < 2:
        return {'RMSE': float('nan'), 'MAE': float('nan'), 'PCC': float('nan'),
                'Spearman': float('nan'), 'R2': float('nan')}
    out = {
        'RMSE':     float(np.sqrt(mean_squared_error(y_true, y_pred))),
        'MAE':      float(np.mean(np.abs(y_true-y_pred))),
        'PCC':      float(stats.pearsonr(y_true, y_pred)[0]),
        'Spearman': float(stats.spearmanr(y_true, y_pred)[0]),
        'R2':       float(r2_score(y_true, y_pred)),
    }
    if drug_ids is not None:
        rs = []
        for d in np.unique(drug_ids):
            mask = drug_ids == d
            if mask.sum() < 2: continue
            yt, yp = y_true[mask], y_pred[mask]
            if np.std(yp) < 1e-8 or np.std(yt) < 1e-8: continue
            try:
                r = float(stats.pearsonr(yt, yp)[0])
                if np.isfinite(r): rs.append(r)
            except Exception: pass
        out['Per-drug PCC'] = float(np.mean(rs)) if rs else float('nan')
    if uncertainties is not None and len(uncertainties)==len(y_true):
        order = np.argsort(uncertainties)
        bins = np.array_split(order, 15)
        ece = sum(abs(np.sqrt(np.mean((y_true[b]-y_pred[b])**2)) - np.sqrt(np.mean(uncertainties[b])))
                  * len(b)/len(y_true) for b in bins if len(b))
        out['ECE'] = float(ece)
    return out

print('Metrics defined.')

---
## 5. Dataset & Training Loop

In [ ]:
# -- 5.1  Dataset and DataLoader ---------------------------------------------
import random
from datetime import timedelta
from torch.utils.data import Dataset, DataLoader

class GDSCDataset(Dataset):
    def __init__(self, df, graph_cache, expr_matrix):
        self.df = df.reset_index(drop=True)
        self.gc = graph_cache
        self.expr_np = expr_matrix.values.astype('float32')
        self.cid2row = {int(cid): i for i, cid in enumerate(expr_matrix.index)}
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        return {'drug_graph': self.gc[int(row['DRUG_ID'])],
                'expr':       self.expr_np[self.cid2row[int(row['COSMIC_ID'])]],
                'y':          float(row['LN_IC50']),
                'drug_id':    int(row['DRUG_ID']),
                'cosmic_id':  int(row['COSMIC_ID'])}

def collate_fn(batch):
    return {
        'drug_batch': Batch.from_data_list([b['drug_graph'] for b in batch]),
        'expr':  torch.tensor(np.stack([b['expr'] for b in batch]), dtype=torch.float),
        'y':     torch.tensor([b['y'] for b in batch], dtype=torch.float),
        'drug_ids':   torch.tensor([b['drug_id'] for b in batch], dtype=torch.long),
        'cosmic_ids': torch.tensor([b['cosmic_id'] for b in batch], dtype=torch.long),
    }

def build_graph_cache(drugs_df):
    smiles_list = drugs_df['SMILES'].dropna().tolist()
    print(f'  Building FG vocab from {len(smiles_list)} SMILES ...')
    fg_vocab = build_fg_vocab(smiles_list)
    cache, failed = {}, 0
    for _, row in tqdm(drugs_df.iterrows(), total=len(drugs_df), desc='  Drug graphs'):
        smi = row['SMILES']
        if pd.isna(smi): failed += 1; continue
        g = smiles_to_graph(smi, fg_vocab=fg_vocab)
        if g: cache[int(row['DRUG_ID'])] = g
        else: failed += 1
    print(f'  Graph cache: {len(cache)} drugs ({failed} failed)')
    return cache, fg_vocab

print('Dataset classes defined.')

In [ ]:
# -- 5.2  Training and evaluation functions ----------------------------------

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def train_one_epoch(model, loader, optimizer, device, scaler=None, grad_clip=1.0):
    model.train(); total_loss = 0.0; n = 0
    use_amp = scaler is not None and device.type == 'cuda'
    for batch in tqdm(loader, desc='  train', leave=False):
        optimizer.zero_grad()
        db = batch['drug_batch'].to(device)
        expr = batch['expr'].to(device)
        y = batch['y'].to(device)
        with torch.amp.autocast(device_type='cuda', enabled=use_amp):
            out = model(drug_batch=db, expr=expr, y=y)
            loss = out['loss']
        if torch.isnan(loss) or torch.isinf(loss):
            print(f'  WARNING: NaN/Inf loss in batch, skipping.')
            optimizer.zero_grad()
            continue
        if use_amp:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            scaler.step(optimizer); scaler.update()
        else:
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step()
        total_loss += loss.item() * len(y); n += len(y)
    return total_loss / n

@torch.no_grad()
def evaluate(model, loader, device, desc='eval', return_preds=False):
    model.eval()
    preds, trues, drug_ids, cosmic_ids, epistemic, aleatoric = [], [], [], [], [], []
    # Disable AMP in evaluation — evidential head needs float32 for numerical stability
    for batch in tqdm(loader, desc=f'  {desc}', leave=False):
        db = batch['drug_batch'].to(device); expr = batch['expr'].to(device)
        with torch.amp.autocast(device_type='cuda', enabled=False):
            out = model(drug_batch=db, expr=expr)
        preds.append(out['pred']['pred'].cpu().numpy())
        trues.append(batch['y'].numpy())
        epistemic.append(out['pred']['epistemic'].cpu().numpy())
        aleatoric.append(out['pred']['aleatoric'].cpu().numpy())
        drug_ids.append(batch['drug_ids'].numpy())
        cosmic_ids.append(batch['cosmic_ids'].numpy())
    y_pred = np.concatenate(preds); y_true = np.concatenate(trues)
    epi = np.concatenate(epistemic); alet = np.concatenate(aleatoric)
    dids = np.concatenate(drug_ids)
    rep = regression_report(y_true, y_pred, drug_ids=dids, uncertainties=epi)
    rep['epistemic_mean'] = float(np.nan_to_num(epi).mean())
    rep['aleatoric_mean'] = float(np.nan_to_num(alet).mean())
    if return_preds:
        rep['_preds'] = {'y_true':y_true,'y_pred':y_pred,'epistemic':epi,'aleatoric':alet,
                         'drug_ids':dids,'cosmic_ids':np.concatenate(cosmic_ids)}
    return rep

print('Training functions defined.')

In [ ]:
# -- 5.3  Main training driver -----------------------------------------------

def train_pathxdrp(cfg, df, expr_matrix, pathway_gene_symbols, label='model'):
    set_seed(cfg['seed'])
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'\n{"="*70}'); print(f'Training {label}  (device={device})')
    print(f'Encoder: drug={cfg["drug_encoder_type"]}  cell={cfg["cell_encoder_type"]}')
    print(f'{"="*70}')

    # Build molecular graph cache
    print('Building graph cache ...')
    drugs_df = df[['DRUG_ID','SMILES']].drop_duplicates()
    graph_cache, _ = build_graph_cache(drugs_df)

    # Pathway gene map (indices)
    gene_list = list(expr_matrix.columns)
    gene_to_idx = {g: i for i, g in enumerate(gene_list)}
    pathway_gene_map = {
        pw: [gene_to_idx[g] for g in genes if g in gene_to_idx]
        for pw, genes in pathway_gene_symbols.items()
        if any(g in gene_to_idx for g in genes)
    }
    n_pairs = sum(len(v) for v in pathway_gene_map.values())
    print(f'Pathway map: {len(pathway_gene_map)} pathways | {n_pairs:,} (pw,gene) pairs')

    # Splits
    tr_idx, va_idx, te_idx = load_split(cfg['split'], cfg['seed'], cfg['fold'])
    print(f'Split sizes: train={len(tr_idx):,} val={len(va_idx):,} test={len(te_idx):,}')

    tr_ds = GDSCDataset(df.iloc[tr_idx], graph_cache, expr_matrix)
    va_ds = GDSCDataset(df.iloc[va_idx], graph_cache, expr_matrix)
    te_ds = GDSCDataset(df.iloc[te_idx], graph_cache, expr_matrix)
    _nw = 2  # Kaggle: 2 workers is safer than 4 to avoid DataLoader hangs
    lk = dict(batch_size=cfg['batch_size'], collate_fn=collate_fn,
              num_workers=_nw, pin_memory=(device.type=='cuda'), persistent_workers=(_nw>0))
    tr_l = DataLoader(tr_ds, shuffle=True,  **lk)
    va_l = DataLoader(va_ds, shuffle=False, **lk)
    te_l = DataLoader(te_ds, shuffle=False, **lk)

    # Model
    sample_g = next(iter(graph_cache.values()))
    node_dim = sample_g.x.size(1)
    edge_dim = sample_g.edge_attr.size(1) if sample_g.edge_attr is not None else 9
    model = PathXDRP(
        node_in_dim=node_dim, edge_in_dim=edge_dim,
        n_genes=expr_matrix.shape[1], pathway_gene_map=pathway_gene_map,
        hidden_dim=cfg['hidden_dim'], n_gat_layers=cfg['n_gat_layers'],
        n_attn_heads=cfg['n_attn_heads'], dropout=cfg['dropout'],
        mask_type=cfg['mask_type'], evidential_lam=cfg['evidential_lam'],
        n_pw_transformer_layers=cfg['n_pw_transformer_layers'],
        drug_encoder_type=cfg['drug_encoder_type'],
        cell_encoder_type=cfg['cell_encoder_type'],
        gene_symbols=gene_list if cfg['cell_encoder_type']=='gene_mamba' else None,
        graph_mamba_kwargs=cfg.get('graph_mamba_kwargs'),
        gene_mamba_kwargs=cfg.get('gene_mamba_kwargs'),
    ).to(device)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'Model parameters: {n_params:,}')

    scaler = torch.amp.GradScaler('cuda') if device.type=='cuda' else None
    opt = torch.optim.AdamW(model.parameters(), lr=cfg['lr'], weight_decay=1e-4)
    wu = torch.optim.lr_scheduler.LinearLR(opt, start_factor=0.1, end_factor=1.0, total_iters=5)
    cos = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(cfg['epochs']-5,1))
    sched = torch.optim.lr_scheduler.SequentialLR(opt, [wu, cos], milestones=[5])

    lam_warmup = cfg.get('lam_warmup_epochs', 50)
    ckpt_dir = WORK / 'checkpoints'; ckpt_dir.mkdir(exist_ok=True)
    ckpt_path = ckpt_dir / f'{label}_{cfg["split"]}_seed{cfg["seed"]}_fold{cfg["fold"]}.pt'

    best_pcc, best_epoch = -float('inf'), 0
    epoch_pbar = tqdm(range(1, cfg['epochs']+1), desc=f'{label} epochs', unit='ep')
    for epoch in epoch_pbar:
        model.evidential_lam = cfg['evidential_lam'] * min(epoch/max(lam_warmup,1), 1.0) if lam_warmup > 0 else cfg['evidential_lam']
        tr_loss = train_one_epoch(model, tr_l, opt, device, scaler)
        va_met  = evaluate(model, va_l, device, desc='val')
        sched.step()
        if va_met['PCC'] > best_pcc:
            best_pcc = va_met['PCC']; best_epoch = epoch
            torch.save(model.state_dict(), ckpt_path)
        _pcc_str = f"{va_met['PCC']:.4f}" if isinstance(va_met['PCC'], float) and not (va_met['PCC'] != va_met['PCC']) else 'NaN'
        epoch_pbar.set_postfix(loss=f"{tr_loss:.4f}", PCC=_pcc_str,
                               best=f"{best_pcc:.4f}@{best_epoch}")

    # Test evaluation
    print('Loading best checkpoint ...')
    model.load_state_dict(torch.load(ckpt_path, map_location=device, weights_only=True))
    te_met = evaluate(model, te_l, device, desc='test', return_preds=True)

    # Save results
    res_dir = WORK / 'results'; res_dir.mkdir(exist_ok=True)
    res_path = res_dir / f'{label}_{cfg["split"]}_seed{cfg["seed"]}_fold{cfg["fold"]}.json'
    preds_path = res_dir / f'{label}_{cfg["split"]}_seed{cfg["seed"]}_fold{cfg["fold"]}_preds.csv'
    result_clean = {k: v for k, v in te_met.items() if not k.startswith('_')}
    with open(res_path, 'w') as f: json.dump(result_clean, f, indent=2)
    p = te_met['_preds']
    pd.DataFrame({'y_true':p['y_true'],'y_pred':p['y_pred'],
                  'epistemic':p['epistemic'],'drug_id':p['drug_ids'],
                  'cosmic_id':p['cosmic_ids']}).to_csv(preds_path, index=False)

    print(f'\n=== {label} Test Results ===')
    for k, v in result_clean.items():
        if not isinstance(v, dict): print(f'  {k:25s}: {v}')
    print(f'Saved: {res_path}')
    return result_clean

print('train_pathxdrp() defined.')

---
## 6. Version A — No Mamba (GAT + PathwaySet)

In [ ]:
CFG_A = dict(
    split            = 'random',   # random | cell_blind | drug_blind | tissue_blind
    seed             = 0,
    fold             = 0,
    drug_encoder_type = 'gat',
    cell_encoder_type = 'pathway_set',
    hidden_dim       = 256,
    n_gat_layers     = 4,
    n_attn_heads     = 8,
    dropout          = 0.1,
    mask_type        = 'soft',
    n_pw_transformer_layers = 1,
    evidential_lam   = 0.01,
    lam_warmup_epochs = 50,
    batch_size       = 256,
    epochs           = 150,
    lr               = 1e-3,
)

results_a = train_pathxdrp(CFG_A, df, expr_matrix, pathway_gene_symbols, label='no_mamba')

---
## 7. Version B — With Mamba (GraphMamba + GeneMamba)

In [ ]:
if not MAMBA_READY:
    print('Skipping Version B  -  mamba_ssm not available.')
    print('Re-run Cell 3 (mamba install) and ensure a CUDA GPU is enabled.')
    results_b = None
else:
    # Pre-download GeneMamba weights
    try:
        print('Pre-downloading GeneMamba backbone (~250 MB) ...')
        _bk = AutoModel.from_pretrained('mineself2016/GeneMamba', trust_remote_code=True)
        print(f'GeneMamba: {sum(p.numel() for p in _bk.parameters()):,} params')
        del _bk; import gc; gc.collect(); torch.cuda.empty_cache()
    except Exception as e:
        print(f'Pre-download failed ({e}); will download automatically during training.')

    CFG_B = dict(
        split            = 'random',
        seed             = 0,
        fold             = 0,
        drug_encoder_type = 'graph_mamba',
        cell_encoder_type = 'gene_mamba',
        hidden_dim       = 256,
        n_gat_layers     = 4,   # unused directly (gm kwargs used)
        n_attn_heads     = 8,
        dropout          = 0.1,
        mask_type        = 'soft',
        n_pw_transformer_layers = 1,
        evidential_lam   = 0.01,
        lam_warmup_epochs = 50,
        batch_size       = 128,   # smaller: backbone uses extra VRAM
        epochs           = 150,
        lr               = 5e-4,
        graph_mamba_kwargs = {'n_gat_layers': 2, 'n_mamba_layers': 2, 'ordering': 'degree'},
        gene_mamba_kwargs  = {'top_k': 2048, 'backbone_id': 'mineself2016/GeneMamba',
                              'freeze_backbone': True},
    )

    results_b = train_pathxdrp(CFG_B, df, expr_matrix, pathway_gene_symbols, label='with_mamba')

---
## 8. Results Comparison & Export

In [ ]:
import zipfile, datetime

rows = []
for label, res in [('No Mamba (GAT+PathwaySet)', results_a),
                   ('With Mamba (GraphMamba+GeneMamba)', results_b)]:
    if res is None: continue
    row = {'Model': label}
    for k in ['PCC','RMSE','R2','Spearman','MAE','ECE','epistemic_mean']:
        row[k] = round(res.get(k, float('nan')), 4)
    rows.append(row)

if rows:
    cmp_df = pd.DataFrame(rows).set_index('Model')
    print('\n=== Final Comparison ===')
    print(cmp_df.to_string())
    cmp_df.to_csv(WORK / 'results' / 'comparison.csv')

# Zip everything for download
ts = datetime.datetime.now().strftime('%Y%m%d_%H%M')
zip_path = WORK / f'pathxdrp_outputs_{ts}.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for pt in (WORK/'checkpoints').glob('*.pt'):
        zf.write(pt, f'checkpoints/{pt.name}')
    for jf in (WORK/'results').glob('*.json'):
        zf.write(jf, f'results/{jf.name}')
    for cf in (WORK/'results').glob('*.csv'):
        zf.write(cf, f'results/{cf.name}')
print(f'\nDownloadable zip: {zip_path}  ({zip_path.stat().st_size/1024**2:.1f} MB)')
print('Go to Kaggle Output tab to download.')